In [1]:
import pandas as pd
import re
import ast
from collections import Counter

import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")

print("spaCy Loaded Successfully")

spaCy Loaded Successfully


In [3]:
jobs = pd.read_csv("../data/jobs_with_skills.csv")

jobs.head()

,job_id,company_name,title,description,location,formatted_work_type,formatted_experience_level,min_salary,max_salary,med_salary,currency,remote_allowed,skills_desc,clean_description,clean_title,extracted_skills
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",Full-time,Not Available,17.0,20.0,0.0,USD,0.0,Requirements: \n\nWe are seeking a College or ...,job descriptiona leading real estate firm in n...,marketing coordinator,[]
1,1829192,Not Available,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",Full-time,Not Available,30.0,50.0,0.0,USD,0.0,Not Available,at aspen therapy and wellness we are committed...,mental health therapist counselor,['excel']
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",Full-time,Not Available,45000.0,65000.0,0.0,USD,0.0,We are currently accepting resumes for FOH - A...,the national exemplar is accepting application...,assitant restaurant manager,[]
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",Full-time,Not Available,140000.0,175000.0,0.0,USD,0.0,This position requires a baseline understandin...,senior associate attorney elder law trusts and...,senior elder law trusts and estates associate ...,['excel']
4,35982263,Not Available,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",Full-time,Not Available,60000.0,80000.0,0.0,USD,0.0,Not Available,looking for hvac service tech with experience ...,service technician,[]


In [9]:
SKILLS = [

    # Programming
    "python","java","c++","c#","javascript","typescript",
    "scala","golang","ruby","php","matlab",

    # Web
    "html","css","react","angular","vue","node.js",
    "django","flask","spring","spring boot",

    # Databases
    "sql","mysql","postgresql","mongodb","oracle",
    "sqlite","redis","snowflake",

    # Data Science
    "pandas","numpy","scikit-learn","tensorflow",
    "pytorch","machine learning","deep learning",
    "data science","data analysis","statistics",

    # Visualization
    "power bi","tableau","matplotlib","seaborn",
    "plotly","excel",

    # Big Data
    "spark","hadoop","databricks","airflow",

    # Cloud
    "aws","azure","gcp","google cloud",

    # DevOps
    "docker","kubernetes","terraform",
    "jenkins","github actions","git",

    # Operating Systems
    "linux","unix",

    # AI
    "nlp","computer vision","generative ai",
    "llm","openai","langchain","rag",

    # Business
    "project management","agile","scrum",
    "communication","leadership",

    # Finance
    "financial analysis","accounting",
    "budgeting","forecasting",

    # Marketing
    "seo","sem","google analytics",
    "content marketing","social media marketing"
]

    

In [10]:
REMOVE_WORDS = [
    "site name",
    "posted date",
    "job description",
    "about the company",
    "business overview",
    "headquartered",
    "organization represents"
]

def remove_metadata(text):
    for word in REMOVE_WORDS:
        text = text.replace(word, "")
    return text.strip()

jobs["clean_description"] = jobs["description"].apply(remove_metadata)

In [11]:
# Quick false-positive check for short/ambiguous tokens
jobs_temp = pd.read_csv("../data/jobs_with_skills.csv")  # old file, just for sampling text

for token in ["r", "sem", "rag"]:
    print(f"\n=== samples containing '{token}' ===")
    matches = jobs_temp["clean_description"].str.contains(
        rf"\b{token}\b", case=False, regex=True, na=False
    )
    for text in jobs_temp.loc[matches, "clean_description"].sample(5, random_state=1):
        print(text[:200], "\n---")


=== samples containing 'r' ===
seeking experienced quality engineer for support in manufacturing processes launching new products and continuous improvement of existing products principal responsibilities provide statistical inform 
---
site name usa california san francisco seattle sixth ave posted date apr 18 2024 the onyx research data platform organization represents a major investment by gsk r d and digital tech designed to deli 
---
job description at boeing we innovate and collaborate to make the world a better place from the seabed to outer space you can contribute to work that matters with a company where diversity equity and  
---
atlas is a nationwide leader in civil engineering materials testing and geotechnical consulting services for environmental industrial and infrastructure construction projects headquartered in austin t 
---
launched in 2020 amidst the covid 19 pandemic i want to mow your lawn r a registered 501 c 3 non profit organization tax id 853447661 offers com

In [8]:
skill_map = {
    "js":"javascript",
    "javascript":"javascript",

    "py":"python",
    "python":"python",

    "postgres":"postgresql",
    "postgresql":"postgresql",

    "google cloud platform":"gcp",
    "google cloud":"gcp",

    "powerbi":"power bi",

    "machine-learning":"machine learning",

    "artificial intelligence":"ai"
}

In [12]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\khush\OneDrive\Desktop\AI-Powered-Job-Market-Intelligence-Platform\AI-Powered-Job-Market-Intelligence-Platform\venv\Scripts\python.exe -m pip install --upgrade pip


In [13]:
import re

# Create regex once
skill_regex = re.compile(
    r"\b(" + "|".join(map(re.escape, SKILLS)) + r")\b",
    re.IGNORECASE
)

def extract_skills(text):

    text = str(text).lower()

    matches = skill_regex.findall(text)

    normalized = [
        skill_map.get(skill, skill)
        for skill in matches
    ]

    return sorted(set(normalized))


from tqdm.auto import tqdm

tqdm.pandas()

jobs["extracted_skills"] = jobs[
    "clean_description"
].progress_apply(extract_skills)

C:\Users\khush\OneDrive\Desktop\AI-Powered-Job-Market-Intelligence-Platform\AI-Powered-Job-Market-Intelligence-Platform\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 123849/123849 [07:59<00:00, 258.04it/s]


In [14]:
jobs["skill_count"] = jobs[
    "extracted_skills"
].apply(len)

jobs["skill_count"].describe()

count    123849.000000
mean          1.608362
std           1.932531
min           0.000000
25%           0.000000
50%           1.000000
75%           2.000000
max          31.000000
Name: skill_count, dtype: float64

In [15]:
jobs[
    ["title","extracted_skills"]
].sample(20)

,title,extracted_skills
105459,"Senior Manager, HRBP","[agile, spark]"
107876,Staff Applications Specialist - Power (Kansas ...,"[communication, power bi, sql]"
22150,Executive Assistant,[communication]
41007,Enterprise Procurement Manager - High Profile ...,"[excel, leadership]"
2290,Facilities Project Manager,[project management]
10536,Installation Engineer,[]
85780,APS-3: Material Coordinator,"[communication, excel]"
120073,Senior Partnership Analytics Analyst,"[agile, data science, python, sql, statistics,..."
56916,Web Developer,"[html, javascript, python, sql]"
120394,Vice President of Population Health Strategy a...,"[communication, leadership]"


In [16]:
jobs["skill_count"].mean()

np.float64(1.6083617954121552)

In [17]:
jobs["skill_count"].max()

np.int64(31)

In [18]:
jobs[["title","extracted_skills"]].sample(5)

,title,extracted_skills
6740,Junior data scientist/java programmer,"[communication, computer vision, data science,..."
21930,Senior Test Engineer,[communication]
51997,Project Manager,"[communication, leadership, project management]"
115345,Cleaning Validation Engineer,[]
40267,Talent Acquisition Specialist,[communication]


In [19]:
jobs.shape

(123849, 17)

In [20]:
zero_skill_jobs = (
    jobs["skill_count"] == 0
).sum()

zero_skill_percentage = (
    zero_skill_jobs / len(jobs) * 100
)

zero_skill_percentage

np.float64(29.47621700619302)

In [21]:
from collections import Counter

skill_counts = Counter(
    skill
    for skills in jobs["extracted_skills"]
    for skill in skills
)

top_skills = pd.DataFrame(
    skill_counts.most_common(20),
    columns=["Skill", "Demand"]
)

top_skills

,Skill,Demand
0,communication,59998
1,leadership,29355
2,excel,18107
3,project management,10229
4,accounting,8801
5,agile,5969
6,sql,5183
7,python,4648
8,data analysis,3239
9,aws,3161


In [22]:
top_skills["Demand_Percentage"] = (
    top_skills["Demand"] / len(jobs) * 100
)

top_skills

,Skill,Demand,Demand_Percentage
0,communication,59998,48.444477
1,leadership,29355,23.702250
2,excel,18107,14.620223
3,project management,10229,8.259251
4,accounting,8801,7.106234
5,agile,5969,4.819579
6,sql,5183,4.184935
7,python,4648,3.752957
8,data analysis,3239,2.615282
9,aws,3161,2.552302


In [23]:
soft_skills = [
    "communication",
    "leadership",
    "project management",
    "agile",
    "scrum"
]

In [24]:
technical_skill_counts = {
    skill: count
    for skill, count in skill_counts.items()
    if skill not in soft_skills
}

In [25]:
top_technical_skills = pd.DataFrame(
    sorted(
        technical_skill_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:20],
    columns=["Skill", "Demand"]
)

top_technical_skills

,Skill,Demand
0,excel,18107
1,accounting,8801
2,sql,5183
3,python,4648
4,data analysis,3239
5,aws,3161
6,forecasting,2976
7,azure,2918
8,java,2664
9,budgeting,2457


In [26]:
top_technical_skills["Demand_Percentage"] = (
    top_technical_skills["Demand"] / len(jobs) * 100
)

top_technical_skills

,Skill,Demand,Demand_Percentage
0,excel,18107,14.620223
1,accounting,8801,7.106234
2,sql,5183,4.184935
3,python,4648,3.752957
4,data analysis,3239,2.615282
5,aws,3161,2.552302
6,forecasting,2976,2.402926
7,azure,2918,2.356095
8,java,2664,2.151006
9,budgeting,2457,1.983867


In [27]:
soft_skill_counts = {
    skill: count
    for skill, count in skill_counts.items()
    if skill in soft_skills
}

top_soft_skills = pd.DataFrame(
    sorted(
        soft_skill_counts.items(),
        key=lambda x: x[1],
        reverse=True
    ),
    columns=["Skill", "Demand"]
)

top_soft_skills

,Skill,Demand
0,communication,59998
1,leadership,29355
2,project management,10229
3,agile,5969
4,scrum,1556


In [28]:
top_soft_skills["Demand_Percentage"] = (
    top_soft_skills["Demand"] / len(jobs) * 100
)

top_soft_skills

,Skill,Demand,Demand_Percentage
0,communication,59998,48.444477
1,leadership,29355,23.702250
2,project management,10229,8.259251
3,agile,5969,4.819579
4,scrum,1556,1.256369


In [29]:
job_skills = jobs[["job_id", "extracted_skills"]].explode("extracted_skills")
job_skills = job_skills.dropna(subset=["extracted_skills"])
job_skills = job_skills.rename(columns={"extracted_skills": "skill"})

job_skills.to_csv("../data/job_skills.csv", index=False)
job_skills.head()

,job_id,skill
1,1829192,communication
3,23221523,communication
5,91700727,communication
5,91700727,data analysis
5,91700727,leadership


In [31]:
sample_golang = jobs[jobs["extracted_skills"].apply(lambda x: "golang" in x)]["clean_description"].sample(5, random_state=1)
for text in sample_golang:
    print(text[:200], "\n---")

Seeking to hire a Sr. Cloud Software Engineer (Hybrid) for a long-term contract in Plano, TX. If you are interested in learning more about this position, email resume to MCasey@calance.com
** We will  
---
Title: Back End EngineerLocation: Riverwoods, IL 60015 (Hybrid)Duration: 6+ Months  Job Description:Summary: As a Back End Engineer, you'll analyze, develop, and design solutions for our application s 
---
Cohesity is a leader in AI-powered data security and management. Aided by an extensive ecosystem of partners, Cohesity makes it easy to secure, protect, manage, and get value from data — across the da 
---
Center 3 (19075), United States of America, McLean, VirginiaSenior Manager, Full Stack Software Engineering - Capital One Software (Remote Eligible)

Capital One has taken a bold journey to build a te 
---
Title: Software EngineerDuration: 18+ monthsLocation: Fully remote, must work PST hoursTarget Pay Range: $85-93/hr
Required Skills & Experience:- 5 plus years' experience in a 

In [32]:
# Find where "go" is defined in your SKILLS list, e.g.:
SKILLS = [
    
    "go", 
    
]

# Change it to:
SKILLS = [
    
    "golang",   # much safer — almost nobody writes "golang" unless they mean the language
    
]

In [33]:

jobs.to_csv(
    "../data/jobs_with_skills_v2.csv",
    index=False
)

In [34]:
jobs.to_csv("../data/jobs_with_skills_v2.csv", index=False)